# Exploration — Mock Data Sprint 1
**Auteur :** Daniel \
**Objectif :** Vérifier la qualité et la cohérence du mock data généré, et poser les premières bases analytiques.

Analyses couvertes :
1. Volumétrie et répartition générale
2. Distribution des commandes (statuts, panier moyen)
3. Taux d'anomalies paiements
4. Répartition géographique des producteurs
5. Top catégories de produits

In [ ]:
import os
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
from sqlalchemy import create_engine, text
from dotenv import load_dotenv

load_dotenv(Path('../../.env'))

db_url = (
    f"postgresql+psycopg2://{os.getenv('POSTGRES_USER')}:{os.getenv('POSTGRES_PASSWORD')}"
    f"@localhost:{os.getenv('POSTGRES_PORT', 5432)}/{os.getenv('POSTGRES_DB')}"
)
engine = create_engine(db_url)

def query(sql):
    with engine.connect() as conn:
        return pd.read_sql(text(sql), conn)

print('Connexion OK')

## 1. Volumétrie générale

In [ ]:
volumetrie = query("""
    SELECT 'users'       AS table_name, COUNT(*) AS nb FROM users
    UNION ALL SELECT 'producers',  COUNT(*) FROM producers
    UNION ALL SELECT 'products',   COUNT(*) FROM products
    UNION ALL SELECT 'orders',     COUNT(*) FROM orders
    UNION ALL SELECT 'order_items',COUNT(*) FROM order_items
    UNION ALL SELECT 'payments',   COUNT(*) FROM payments
""")
volumetrie

## 2. Répartition des statuts de commande

In [ ]:
statuts = query("""
    SELECT status, COUNT(*) AS nb,
           ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 1) AS pct
    FROM orders GROUP BY status ORDER BY nb DESC
""")

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(statuts['status'], statuts['nb'], color='steelblue')
ax.set_title('Répartition des statuts de commande')
ax.set_ylabel('Nombre de commandes')
for i, row in statuts.iterrows():
    ax.text(i, row['nb'] + 20, f"{row['pct']}%", ha='center')
plt.tight_layout()
plt.show()
statuts

## 3. Distribution du panier moyen

In [ ]:
paniers = query("""
    SELECT total_amount FROM orders WHERE status != 'cancelled'
""")

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(paniers['total_amount'], bins=40, color='steelblue', edgecolor='white')
ax.axvline(paniers['total_amount'].mean(), color='red', linestyle='--', label=f"Moyenne : {paniers['total_amount'].mean():.2f}€")
ax.axvline(paniers['total_amount'].median(), color='orange', linestyle='--', label=f"Médiane : {paniers['total_amount'].median():.2f}€")
ax.set_title('Distribution du panier moyen')
ax.set_xlabel('Montant (€)')
ax.legend()
plt.tight_layout()
plt.show()

## 4. Taux d'anomalies paiements

In [ ]:
paiements = query("""
    SELECT status, is_simulated_error, COUNT(*) AS nb
    FROM payments
    GROUP BY status, is_simulated_error
    ORDER BY status
""")

taux_echec = query("""
    SELECT
        ROUND(SUM(CASE WHEN status = 'failed' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS taux_echec_pct
    FROM payments
""")

print(f"Taux d'échec global : {taux_echec['taux_echec_pct'].iloc[0]}% (cible : 5%)")
paiements

## 5. Top catégories par chiffre d'affaires

In [ ]:
ca_categories = query("""
    SELECT p.category,
           COUNT(DISTINCT o.id)                          AS nb_commandes,
           ROUND(SUM(oi.quantity * oi.unit_price_snapshot)::numeric, 2) AS ca_total
    FROM order_items oi
    JOIN products p ON p.id = oi.product_id
    JOIN orders o   ON o.id = oi.order_id
    WHERE o.status IN ('confirmed', 'shipped', 'delivered')
    GROUP BY p.category
    ORDER BY ca_total DESC
""")

fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(ca_categories['category'], ca_categories['ca_total'], color='steelblue')
ax.set_title('Chiffre d\'affaires par catégorie')
ax.set_xlabel('CA total (€)')
plt.tight_layout()
plt.show()
ca_categories

## 6. Répartition géographique des producteurs

In [ ]:
geo = query("""
    SELECT location_region, COUNT(*) AS nb_producteurs
    FROM producers
    GROUP BY location_region
    ORDER BY nb_producteurs DESC
""")

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(geo['location_region'], geo['nb_producteurs'], color='steelblue')
ax.set_title('Producteurs par région')
ax.set_ylabel('Nombre de producteurs')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()
geo